In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import folium

# === 1. GENERATE DATA KRIMINALITAS JAKARTA ===
np.random.seed(99)

def create_crime_zone(lat, lon, n, spread):
    lats = np.random.normal(lat, spread, n)
    lons = np.random.normal(lon, spread, n)
    return np.stack((lats, lons), axis=-1)

# Lokasi Rawan (Titik Pusat Cluster)
kota_tua = create_crime_zone(-6.1352, 106.8133, 120, 0.004)   # Copet Wisata
tanah_abang = create_crime_zone(-6.1893, 106.8146, 150, 0.005)# Kejahatan Pasar
pulogadung = create_crime_zone(-6.1852, 106.8906, 80, 0.006)  # Preman Terminal
blok_m = create_crime_zone(-6.2442, 106.8005, 60, 0.003)      # Area Hiburan

# Kejahatan Random (Noise) di seluruh Jakarta
random_crimes = np.random.uniform(low=[-6.30, 106.70], high=[-6.10, 106.95], size=(60, 2))

data_crime = np.vstack([kota_tua, tanah_abang, pulogadung, blok_m, random_crimes])
df = pd.DataFrame(data_crime, columns=['lat', 'lon'])

print(f"Laporan Masuk: {len(df)} insiden kejahatan tercatat.")

Laporan Masuk: 470 insiden kejahatan tercatat.


In [ ]:
# === 2. ANALISIS DENSITAS ===

coords_rad = np.radians(df[['lat', 'lon']].to_numpy())

# Parameter Polisi:
# Radius 0.5 KM (500 meter)
# Minimal 10 laporan kejadian
epsilon_km = 0.5
min_samples = 10

db = DBSCAN(eps=epsilon_km/6371.0088, min_samples=min_samples, metric='haversine', algorithm='ball_tree')
db.fit(coords_rad)

df['cluster'] = db.labels_

# Hitung statistik
n_hotspots = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print(f"Sistem mendeteksi {n_hotspots} ZONA MERAH yang butuh patroli rutin.")

Sistem mendeteksi 4 ZONA MERAH yang butuh patroli rutin.


In [ ]:
# === 3. DASHBOARD PETA KRIMINALITAS ===

# Peta Dasar Gelap (Keren untuk presentasi crime/night data)
m = folium.Map(location=[-6.1754, 106.8272], zoom_start=12, tiles='CartoDB dark_matter')

# Warna: Cluster = Merah Menyala, Noise = Putih Pudar
for idx, row in df.iterrows():
    cluster_id = row['cluster']

    if cluster_id == -1:
        # INSIDEN BIASA (Random) -> Titik Putih Kecil
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=2,
            color='#bdc3c7', # Abu-abu terang
            fill=True,
            fill_opacity=0.3,
            popup="Insiden Tunggal"
        ).add_to(m)
    else:
        # ZONA BAHAYA -> Lingkaran Merah dengan efek 'Pulse' (Simulasi)
        # Kita buat lingkaran agak besar transparan untuk menunjukkan 'zona'
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=8,
            color='#e74c3c', # Merah
            fill=True,
            fill_color='#c0392b',
            fill_opacity=0.6,
            popup=f"ZONA RAWAN #{int(cluster_id)}"
        ).add_to(m)

title_html = '''
             <h3 align="center" style="font-size:16px; color:white"><b>⚠️ ANALYSIS CRIME HOTSPOTS (JAKARTA)</b></h3>
             '''
m.get_root().html.add_child(folium.Element(title_html))

m